<a href="https://colab.research.google.com/github/JariusF10-8/bt4222/blob/main/notebooks/models/04c_UBCF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline: user-based collaborative filtering (UBCF)

**Purpose.** UBCF tests whether users with similar click histories click the same news. It uses only user IDs, article IDs and training clicks: no text, no entities and no engineered features. It is the neighbourhood counterpart to the ID-only NeuMF baseline (`04a_NeuMF.ipynb`).

**How a candidate is scored**

1. Build a binary user by article matrix from training clicks (a repeated click counts once).
2. For the impression's user, find the 50 most similar training users by cosine similarity, excluding the user.
3. The candidate's score is the similarity-weighted share of those neighbours who clicked it.
4. A user with no training clicks gets the article's training popularity instead. An article never clicked in training scores 0.

**Data.** Reads the shared preprocessing build (`data/processed/v2`) only. It does not need the feature bundle.

**Leakage rule.** Each training day is scored with a model fitted on earlier training days only. Validation and test are scored with one model fitted on all training clicks, so no validation or test click is ever used to score.

## 1. Setup

In Colab, add the shared Drive folder `BT4222 Group Project` to My Drive first (see the README). The cell mounts Drive and reads `data/processed/v2` from it. Locally, it reads the same path inside this repository. Scores and metrics are written to `outputs/models/ubcf_v1`.

In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import NearestNeighbors

STORAGE_ROOT = None           # None: the shared Drive folder in Colab, this repository locally
PROCESSED_VERSION = 'v2'      # shared preprocessing build
TOP_K = 50                    # neighbours per user, fixed before looking at test
CANDIDATE_CHUNK_SIZE = 100_000

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_repo():
    configured = os.environ.get('BT4222_PROJECT_ROOT')
    options = ([Path(configured).expanduser()] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    return next((p.resolve() for p in options if (p / 'src' / 'preprocessing.py').is_file()), None)


if STORAGE_ROOT is None:
    STORAGE_ROOT = Path('/content/drive/MyDrive/BT4222 Group Project') if IN_COLAB else find_repo()
if STORAGE_ROOT is None:
    raise FileNotFoundError('Open this notebook inside the bt4222 repository, or set STORAGE_ROOT.')
STORAGE_ROOT = Path(STORAGE_ROOT)
PROCESSED_DIR = Path(os.environ.get('BT4222_PROCESSED_DIR', STORAGE_ROOT / 'data/processed' / PROCESSED_VERSION))
if not (PROCESSED_DIR / 'manifest.json').is_file():
    raise FileNotFoundError(f'Preprocessing manifest missing: {PROCESSED_DIR}. '
                            'In Colab, add the shared Drive folder to My Drive. Locally, copy data/processed/v2 from Drive.')
OUTPUT_DIR = STORAGE_ROOT / 'outputs/models/ubcf_v1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def shown(path):
    if IN_COLAB:
        return str(path)
    try:
        return '<repo>/' + str(Path(path).relative_to(STORAGE_ROOT))
    except ValueError:
        return str(path)


print('Environment:', 'Colab' if IN_COLAB else 'local')
print('Preprocessed input:', shown(PROCESSED_DIR))
print('Output folder:', shown(OUTPUT_DIR))
print('TOP_K:', TOP_K)

Environment: local
Preprocessed input: <repo>/data/processed/v2
Output folder: <repo>/outputs/models/ubcf_v1
TOP_K: 50


## 2. Load the partitions and check impression integrity

Every impression keeps its complete original candidate list. Labels are only used as targets and for evaluation.

In [2]:
PARTITIONS = ('train', 'validation', 'test')
def load_partition(part):
    folder = PROCESSED_DIR / part
    events = pd.read_csv(folder / 'events.csv', usecols=['event_key','user_id','timestamp','history_ids'],
                         dtype={'event_key':str,'user_id':str,'history_ids':str})
    events['timestamp'] = pd.to_datetime(events['timestamp'], errors='raise')
    candidates = pd.read_csv(folder / 'candidates.csv',
                             usecols=['event_key','candidate_position','news_id','label'],
                             dtype={'event_key':str,'candidate_position':'int32','news_id':str,'label':'int8'})
    assert events.event_key.is_unique
    assert not candidates.duplicated(['event_key','candidate_position']).any()
    assert candidates.label.isin([0,1]).all()
    assert candidates.event_key.isin(events.event_key).all()
    return events, candidates

data = {part:load_partition(part) for part in PARTITIONS}
for part,(events,candidates) in data.items():
    print(part, 'events:',len(events),'candidates:',len(candidates),
          'dates:',events.timestamp.min(),events.timestamp.max())

train events: 95071 candidates: 3389342 dates: 2019-11-09 00:00:19 2019-11-12 23:59:25
validation events: 61894 candidates: 2454102 dates: 2019-11-13 00:01:00 2019-11-14 23:59:13
test events: 73152 candidates: 2740998 dates: 2019-11-15 00:00:01 2019-11-15 23:58:03


## 3. The UBCF model

`fit` builds the click matrix, the training popularity used as the fallback, and a brute-force cosine nearest-neighbour index. `score_events` scores all candidates of one user at a time, so each neighbourhood is computed once per user. The small synthetic check confirms that a candidate clicked by a similar user scores higher and that the user is not counted as their own neighbour.

One thread is used for each single-user neighbour query, which is faster than splitting such a small query across threads. The neighbours found are the same either way.

In [3]:
train_events, train_candidates = data['train']
train_events = train_events[['event_key','user_id','timestamp']].copy()
train_events['day'] = train_events.timestamp.dt.normalize()
train_clicks = train_candidates.loc[train_candidates.label.eq(1), ['event_key','news_id']].merge(
    train_events, on='event_key', how='left', validate='many_to_one')
assert train_clicks.timestamp.notna().all()

class UBCF:
    def __init__(self, k=50):
        self.k = k

    def fit(self, clicks):
        self.user_ids = pd.Index(sorted(clicks.user_id.unique()))
        self.article_ids = pd.Index(sorted(clicks.news_id.unique()))
        self.user_index = pd.Series(np.arange(len(self.user_ids)), index=self.user_ids)
        self.article_index = pd.Series(np.arange(len(self.article_ids)), index=self.article_ids)
        u = self.user_index.loc[clicks.user_id].to_numpy(dtype=np.int32)
        a = self.article_index.loc[clicks.news_id].to_numpy(dtype=np.int32)
        mat = sparse.csr_matrix((np.ones(len(u),dtype=np.float32),(u,a)),
                                shape=(len(self.user_ids),len(self.article_ids)))
        mat.data[:] = 1.0  # binary implicit feedback; repeated clicks count once
        self.matrix = mat
        self.popularity = np.asarray(mat.sum(axis=0)).ravel().astype(np.float32)
        self.popularity /= max(float(self.popularity.max(initial=0)),1.0)
        self.neighbors = None
        if len(self.user_ids) > 1 and self.matrix.nnz:
            self.neighbors = NearestNeighbors(metric='cosine', algorithm='brute',
                                              n_neighbors=min(self.k+1,len(self.user_ids)),n_jobs=1)
            self.neighbors.fit(mat)
        return self

    def score_events(self, events, candidates):
        # Group by user to avoid recomputing the same neighborhood for every candidate.
        users = events[['event_key','user_id']].set_index('event_key')['user_id']
        out = np.empty(len(candidates), dtype=np.float32)
        event_keys = candidates.event_key.to_numpy()
        candidate_users = users.reindex(event_keys)
        assert candidate_users.notna().all()
        candidate_users = candidate_users.to_numpy()
        article_pos = self.article_index.reindex(candidates.news_id).to_numpy()
        known_article = pd.notna(article_pos)
        # Candidate row positions for each user, found in one pass.
        user_positions = pd.Series(np.arange(len(candidates))).groupby(candidate_users, sort=False).indices
        for uid, positions in user_positions.items():
            known = positions[known_article[positions]]
            out[positions] = 0.0
            if len(known) == 0:
                continue
            cols = article_pos[known].astype(np.int32)
            # Fallback is training-only popularity, including unseen users.
            scores = self.popularity[cols].copy()
            if uid in self.user_index.index and self.neighbors is not None:
                row_id = int(self.user_index.loc[uid])
                distances, indices = self.neighbors.kneighbors(self.matrix[row_id],return_distance=True)
                neighbor_ids = indices[0]
                weights = np.clip(1.0 - distances[0],0,1)
                mask = neighbor_ids != row_id
                neighbor_ids, weights = neighbor_ids[mask], weights[mask]
                valid = weights > 0
                neighbor_ids, weights = neighbor_ids[valid], weights[valid]
                if len(neighbor_ids):
                    neighbor_clicks = self.matrix[neighbor_ids][:,cols]
                    weighted = np.asarray(neighbor_clicks.T @ weights).ravel()
                    scores = (weighted / weights.sum()).astype(np.float32)
            out[known] = scores
        return out

# Synthetic check: shared clicks identify similar users, self is excluded.
toy = pd.DataFrame({'user_id':['u1','u1','u2','u2','u3'],
                    'news_id':['a','b','a','c','d']})
toy_model = UBCF(k=2).fit(toy)
toy_e = pd.DataFrame({'event_key':['e'],'user_id':['u1']})
toy_c = pd.DataFrame({'event_key':['e','e'],'news_id':['c','d']})
assert toy_model.score_events(toy_e,toy_c)[0] > toy_model.score_events(toy_e,toy_c)[1]
print('Synthetic neighbor test passed.')

Synthetic neighbor test passed.


## 4. Score every candidate

- **Train:** each day is scored by a model fitted on the earlier training days. The first day (9 November) has no earlier clicks, so all its scores are 0.
- **Validation and test:** one model fitted on all training clicks scores every candidate.

Scores are saved with the original candidate order and labels.

In [4]:
results = {}
for part,(events,candidates) in data.items():
    if part == 'train':
        dates = events.timestamp.dt.normalize()
        scores = np.zeros(len(candidates),dtype=np.float32)
        for day in sorted(dates.unique()):
            event_subset = events.loc[dates.eq(day)]
            selected = candidates.event_key.isin(event_subset.event_key)
            click_subset = train_clicks.loc[train_clicks.day.lt(day)]
            if click_subset.empty:
                # No historical feedback exists for the first training day.
                scores[selected.to_numpy()] = 0.0
            else:
                model = UBCF(TOP_K).fit(click_subset)
                scores[selected.to_numpy()] = model.score_events(event_subset,candidates.loc[selected])
            print('train day:',day,'candidates:',int(selected.sum()),'past clicks:',len(click_subset))
    else:
        if 'heldout_model' not in globals():
            heldout_model = UBCF(TOP_K).fit(train_clicks)
        scores = np.empty(len(candidates),dtype=np.float32)
        for start in range(0,len(candidates),CANDIDATE_CHUNK_SIZE):
            batch = candidates.iloc[start:start+CANDIDATE_CHUNK_SIZE]
            scores[start:start+len(batch)] = heldout_model.score_events(events,batch)
    assert np.isfinite(scores).all() and (scores>=0).all() and (scores<=1.00001).all()
    result = candidates.copy()
    result['ubcf_score'] = scores
    result.to_csv(OUTPUT_DIR / f'{part}_ubcf_scores.csv',index=False)
    results[part] = result
    print(part,'saved:',len(result))

train day: 2019-11-09 00:00:00 candidates: 524529 past clicks: 0


train day: 2019-11-10 00:00:00 candidates: 520041 past clicks: 20179


train day: 2019-11-11 00:00:00 candidates: 1166308 past clicks: 41953


train day: 2019-11-12 00:00:00 candidates: 1178464 past clicks: 90030


train saved: 3389342


validation saved: 2454102


test saved: 2740998


## 5. Quick evaluation

A first check of the saved scores. Here MRR uses the first click only and ties keep the original candidate order, so these numbers are not on exactly the same footing as the other models. Section 6 gives the comparable numbers.

In [5]:
from sklearn.metrics import roc_auc_score

def evaluate(frame):
    rows=[]
    for _,group in frame.groupby('event_key',sort=False):
        y=group.label.to_numpy(dtype=np.int8)
        s=group.ubcf_score.to_numpy(dtype=np.float64)
        order=np.argsort(-s,kind='stable')
        ranked=y[order]
        positives=int(y.sum())
        if not positives: continue
        ranks=np.flatnonzero(ranked)+1
        first=int(ranks[0])
        row={'MRR':1/first}
        for k in (5,10):
            dcg=sum(float(ranked[i])/np.log2(i+2) for i in range(min(k,len(y))))
            ideal=sum(1/np.log2(i+2) for i in range(min(k,positives)))
            row[f'nDCG@{k}']=dcg/ideal if ideal else np.nan
        row['AUC']=roc_auc_score(y,s) if 0<positives<len(y) else np.nan
        rows.append(row)
    return pd.DataFrame(rows).mean().to_dict(),len(rows)

summary=[]
for part,frame in results.items():
    metrics,n=evaluate(frame)
    summary.append({'partition':part,'eligible_impressions':n,**metrics})
summary=pd.DataFrame(summary)
print(summary.to_string(index=False))
summary.to_csv(OUTPUT_DIR / 'metrics.csv',index=False)

 partition  eligible_impressions      MRR   nDCG@5  nDCG@10      AUC
     train                 95071 0.286645 0.269584 0.325060 0.511799
validation                 61894 0.242882 0.218855 0.276722 0.504631
      test                 73152 0.241756 0.220297 0.282261 0.495622


## 6. Final test result: the same metrics as the other models

This cell uses the metric function and tie order of the NeuMF, CBF and MLP notebooks (04a, 04b, 05a and 05b), so UBCF can be compared directly with them. Per impression it computes AUC, MRR over all clicked candidates, nDCG@5 and nDCG@10, then averages. Ties are broken by a fixed random order (seed 2026) that does not depend on the labels. The random-order rows reproduce the floors reported in the other notebooks (0.2703 on validation, 0.2865 on test).

In [6]:
TIE_SEED = 2026  # the same label-independent tie order as the NeuMF, CBF and MLP notebooks (04a, 04b, 05a and 05b)


def ranking_metrics(event_keys, y, scores, tie):
    """Shared MIND-style metrics from 04b_CBF.ipynb: per impression, then averaged.

    MRR averages the reciprocal ranks of all clicked candidates. Ties are broken by a fixed
    random order that does not depend on the labels. Impressions need a click and a non-click.
    """
    data = pd.DataFrame({'g': pd.factorize(event_keys)[0], 'y': y.astype('int64'),
                         's': scores.astype('float64'), 'tie': tie})
    data = data.sort_values(['g', 's', 'tie'], ascending=[True, False, False], kind='stable')
    data['rank'] = data.groupby('g', sort=False).cumcount() + 1
    data['avg_rank'] = data.groupby('g', sort=False)['s'].rank(method='average', ascending=True)
    group = data.groupby('g', sort=False)
    n, pos = group['y'].size(), group['y'].sum()
    valid = (pos > 0) & (pos < n)
    data['rr'] = data['y'] / data['rank']
    data['gain'] = data['y'] / np.log2(data['rank'] + 1)
    data['pos_rank'] = data['y'] * data['avg_rank']
    sums = data.groupby('g', sort=False).agg(rr=('rr', 'sum'), pr=('pos_rank', 'sum'))
    table = pd.DataFrame({'AUC': (sums['pr'] - pos * (pos + 1) / 2) / (pos * (n - pos)),
                          'MRR': sums['rr'] / pos})
    discount = np.r_[0., np.cumsum(1 / np.log2(np.arange(2, 12)))]
    for k in (5, 10):
        dcg = data.loc[data['rank'].le(k)].groupby('g', sort=False)['gain'].sum().reindex(n.index, fill_value=0.)
        table[f'nDCG@{k}'] = dcg / discount[np.minimum(pos.to_numpy(dtype=int), k)]
    return {'impressions': int(valid.sum()), **table.loc[valid].mean().to_dict()}


shared = []
for part, frame in results.items():
    y = frame['label'].to_numpy()
    tie = np.random.default_rng(TIE_SEED).random(len(y))
    keys = frame['event_key'].to_numpy()
    shared.append({'partition': part, 'model': 'UBCF', **ranking_metrics(keys, y, frame['ubcf_score'].to_numpy(), tie)})
    shared.append({'partition': part, 'model': 'random order', **ranking_metrics(keys, y, np.zeros(len(y)), tie)})
shared = pd.DataFrame(shared)
shared.to_csv(OUTPUT_DIR / 'metrics_shared_protocol.csv', index=False)
display(shared.round(4))

,partition,model,impressions,AUC,MRR,nDCG@5,nDCG@10
0,train,UBCF,95071,0.5118,0.2577,0.2711,0.3263
1,train,random order,95071,0.5000,0.2426,0.2525,0.3082
2,validation,UBCF,61894,0.5046,0.2159,0.2180,0.2764
3,validation,random order,61894,0.5000,0.2109,0.2119,0.2703
4,test,UBCF,73152,0.4956,0.2165,0.2201,0.2828
5,test,random order,73152,0.5000,0.2182,0.2229,0.2865


**Final test result** (the number used in the report): UBCF on all 73,152 test impressions scores AUC 0.4956, MRR 0.2165, nDCG@5 0.2201 and **nDCG@10 0.2828**, slightly below a random order (0.2865).

## 7. How often UBCF can score

The share of candidates scored 0 and of impressions where every candidate has the same score show how much of each slate the neighbourhood can rank. Tied impressions fall back to the original candidate order.

In [7]:
rows = []
for part, frame in results.items():
    scores = frame['ubcf_score'].to_numpy()
    tied = frame.groupby('event_key', sort=False)['ubcf_score'].nunique().eq(1)
    rows.append({'partition': part,
                 'candidates': len(frame),
                 'share scored 0': round(float((scores == 0).mean()), 4),
                 'impressions with every candidate tied': round(float(tied.mean()), 4)})
coverage = pd.DataFrame(rows)
display(coverage)
coverage.to_csv(OUTPUT_DIR / 'score_coverage.csv', index=False)

,partition,candidates,share scored 0,impressions with every candidate tied
0,train,3389342,0.7989,0.5124
1,validation,2454102,0.9251,0.7005
2,test,2740998,0.9310,0.5036


## 8. Verify the exports

The saved score files must have the same rows, order and labels as the input candidates, with finite scores between 0 and 1.

In [8]:
for part,(_,original) in data.items():
    exported=pd.read_csv(OUTPUT_DIR / f'{part}_ubcf_scores.csv',
                         dtype={'event_key':str,'news_id':str,'candidate_position':'int32','label':'int8'})
    assert len(exported)==len(original)
    assert exported[['event_key','candidate_position','news_id','label']].equals(
        original[['event_key','candidate_position','news_id','label']])
    assert exported.ubcf_score.between(0,1.00001).all()
    assert exported.ubcf_score.notna().all()
print('All exported partitions verified.')

All exported partitions verified.


## Findings

This run used a local rebuild of the shared `v2` files (the same row counts and sample rows as the Drive copy) on a 2-core CPU and took about 10 minutes. In Colab the setup cell reads `v2` from Drive.

Results with the shared metrics (Section 6):

| Partition | AUC | MRR | nDCG@5 | nDCG@10 | Random order nDCG@10 |
|---|---|---|---|---|---|
| Train (each day scored from earlier days) | 0.5118 | 0.2577 | 0.2711 | 0.3263 | 0.3082 |
| Validation | 0.5046 | 0.2159 | 0.2180 | 0.2764 | 0.2703 |
| **Test** | **0.4956** | **0.2165** | **0.2201** | **0.2828** | 0.2865 |

1. **UBCF is at the random level.** It beats a random order by only 0.006 nDCG@10 on validation and is 0.004 below it on test.
2. **Most candidates cannot be scored.** 93.1% of test candidates score 0, and in 50.4% of test impressions every candidate has the same score. Almost 90% of test exposures come from users with no training clicks, and almost 90% are articles never shown in training, so there are no neighbours and no click history to use.
3. **The popularity fallback works against fresh news.** The 6.9% of test candidates that UBCF can score are mostly older articles with training clicks, and they are clicked less often (2.06%) than the candidates it cannot score (4.21%). Ranking them first is why test AUC falls just below 0.5.
4. **The training row is not in-sample.** Each training day is scored only from earlier days, and 9 November has no earlier clicks, so its candidates all score 0.

**Limitation.** `TOP_K = 50` was fixed in advance and not tuned on validation.